# Lab 1 · Setup: your GPU workbench

**Time:** 15 min · **You will:** connect to your Runpod GPU, install the course packages, download the model, and get a first answer from a small language model (SLM).

Throughout the course you'll build one thing step by step: **an HR assistant for ABC Company** (a fictional company) that answers employees' policy questions correctly, admits when it doesn't know, and runs fast and cheap on a single GPU.

| Day | Labs | You'll be able to… |
|---|---|---|
| **1** | 1 to 8 | see how a model reads text; explain attention; build and measure a RAG assistant; protect it with guardrails (rules and a model-based judge) |
| **2** | 9 to 15 | build an agent that acts: tools and ReAct with LangChain, LangGraph workflows, memory, guardrails with NeMo Guardrails, evals with DeepEval, and a full dialogue manager |
| **3** | 16 to 21 | make it fast, small and cheap to serve: KV cache, quantisation, batching, speculative decoding, capacity, and a capstone |

**How the notebooks work:** all the code is already written, so you never have to fix code.
1. Click **Kernel → Restart Kernel and Run All Cells**, then scroll down and read each result.
2. In a **🔧 Try it** cell, change only the value marked `🔧` (a sentence in quotes or a number) and press **Shift+Enter** to run that cell again. See what changes.
3. A **✅** line explains the result. A **⚠️** line means the result was unexpected — usually because of a 🔧 change. Put the original value back and run again.
4. **👀 See it** cells have nothing to change: run them and read.

Run the labs **in order within each day**: later labs build on the ideas of the earlier ones. Optional labs for more depth are in the `extras/` folder.

## 1.1 Install the course packages
Run once per Pod. It takes **3 to 6 minutes** and prints a lot of text; yellow warnings are normal. (After you stop and start a Pod the packages are erased: run this cell again.)

In [ ]:
import os
if os.environ.get("SLM_SKIP_INSTALL") != "1":
    _course = next(d for d in ("..", "../..") if os.path.exists(os.path.join(d, "requirements.txt")))
    %pip install -q -r {_course}/requirements.txt

## 1.2 Load the course helpers and check the hardware
`slm_lab.py` holds the boilerplate (loading, a `chat()` wrapper, timing, scoring). Open it once to see what's inside.

In [ ]:
import sys, os
# find the course folder (the one holding slm_lab.py), whichever folder this notebook was opened from
sys.path.insert(0, next(os.path.abspath(d) for d in ("..", "../..") if os.path.exists(os.path.join(d, "slm_lab.py"))))
from slm_lab import *

gpu_report()

### 🔧 Try it 1.1 — How big a model fits?
A model's weights need **parameters × bytes per parameter** of GPU memory. In bfloat16 every parameter takes **2 bytes**. We also keep **20 %** of the memory free for the work the model does while answering.

The cell works out how many **billion** parameters fit on your GPU.

🔧 **Try:** change `bytes_per_param` to `0.5` (a 4-bit model) or `keep_free` to `0.5`, and run the cell again.

In [ ]:
info = gpu_report()
vram_gb = info.get("vram_gb", 16)      # on a CPU-only machine we assume 16 GB

bytes_per_param = 2      # 🔧 2 = bfloat16 · 1 = 8-bit · 0.5 = 4-bit
keep_free = 0.20         # 🔧 share of memory kept free (0.20 = 20 %)

max_billion_params = vram_gb * (1 - keep_free) / bytes_per_param
print(f"About {max_billion_params:.1f} B parameters fit in {vram_gb} GB")

In [ ]:
check(abs(max_billion_params - vram_gb * (1 - keep_free) / bytes_per_param) < 0.05,
      f"A {vram_gb} GB GPU holds roughly a {max_billion_params:.0f}B model at {bytes_per_param} bytes per parameter. "
      "Our 0.5B model uses a small slice of it - room to train.",
      "usable memory = vram_gb × (1 - keep_free); 1 GB holds 1 ÷ bytes_per_param billion parameters.")

## 1.3 Download the models
We use **Qwen2.5-0.5B-Instruct** (an SLM with ~0.5 billion parameters, ~1 GB) for most labs, a tiny embedding model for search in Lab 6, and the larger **Qwen2.5-3B-Instruct** (~6 GB) for the labs where answer quality matters (Labs 6, 7, 13 and 21). We download the 3B model now so it does not delay you later: this is the longest download of the course. On Runpod they're cached under `/workspace`, so they survive stopping the Pod.

In [ ]:
from huggingface_hub import snapshot_download
for repo in (MODEL_ID, EMBED_ID, "Qwen/Qwen2.5-3B-Instruct"):
    path = snapshot_download(repo, allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model"])
    print("✅", repo, "→", path)

## 1.4 First conversation

In [ ]:
tok, model = load()
print(f"{MODEL_ID} on {model.device}, {model_size_mb(model):,.0f} MB of weights")
with Timer() as t:
    answer = chat(model, tok, "In one sentence, what is a small language model?")
print(answer, f"\n\n({t.seconds:.1f} s)")

### 🔧 Try it 1.2 — Ask it about ABC Company
ABC Company's real answer is **20 days** of paternity leave (Section 4.3 of its HR policy). Read the model's reply carefully. Is it right? Does it *sound* sure?

🔧 **Try:** change `question` to another ABC question, e.g. *"How many sick days do ABC Company employees get?"* (the real answer is 12).

In [ ]:
question = "How many days of paternity leave do ABC Company employees get?"   # 🔧 ask anything

reply = chat(model, tok, question)
print(reply)

In [ ]:
check(isinstance(reply, str) and len(reply) > 0, "Got a reply. Keep it in mind - Lab 5 measures this properly.",
      "the model returned nothing - run the cell again.")

The model has never seen ABC Company's policy. Depending on the wording it either **invents** a number (a fluent, confident **hallucination**) or **dodges** with generic advice — neither helps an employee. Fixing that is the thread through this whole course.

**Next:** Lab 2 — what the model actually sees when you type a question (short, or homework). Then Lab 3.